In [ ]:

from pathlib import Path
from docling.document_converter import DocumentConverter


In [ ]:

# Initialize the converter
converter = DocumentConverter()

PDF_PATH = Path(r"C:\Programming\multi-agent-system\data\uploads\pdf\Angebot_test.pdf")

# Convert a local file
result = converter.convert(PDF_PATH)

# Export the result to Markdown
markdown_output = result.document.export_to_markdown()

# Save Markdown file
markdown_path = Path(r"C:\Programming\multi-agent-system\data\uploads\markdown\angebot_test_output.md")
markdown_path.write_text(markdown_output, encoding="utf-8")

print(f"Markdown saved in: {markdown_path.resolve()}")

[INFO] 2026-10-06 06:25:45,206 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-06 06:25:45,247 [RapidOCR] download_file.py:60: File exists and is valid: C:\Programming\multi-agent-system\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-10-06 06:25:45,248 [RapidOCR] main.py:63: Using C:\Programming\multi-agent-system\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-10-06 06:25:45,376 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-06 06:25:45,392 [RapidOCR] download_file.py:60: File exists and is valid: C:\Programming\multi-agent-system\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-06 06:25:45,392 [RapidOCR] main.py:63: Using C:\Programming\multi-agent-system\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-10-06 06:25:45,470 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-10-06 06:25:45,51

In [ ]:

# Save Markdown file
markdown_path = Path(r"C:\Programming\multi-agent-system\data\uploads\markdown\angebot_test_output.md")
markdown_path.write_text(markdown_output, encoding="utf-8")

print(f"Markdown saved in: {markdown_path.resolve()}")

Markdown gespeichert unter: C:\Programming\multi-agent-system\data\uploads\markdown\angebot_test_output.md


In [ ]:
from docling.document_converter import DocumentConverter
from utils.sitemap import get_sitemap_urls

converter = DocumentConverter()

# --------------------------------------------------------------
# Basic PDF extraction
# --------------------------------------------------------------

result = converter.convert("https://arxiv.org/pdf/2408.09869")

document = result.document
markdown_output = document.export_to_markdown()
json_output = document.export_to_dict()

print(markdown_output)

In [ ]:

# Markdown-Datei speichern
markdown_path = Path("gpt4_1_mini_rechnung_output.md")
markdown_path.write_text(markdown_output, encoding="utf-8")

print(f"Markdown gespeichert unter: {markdown_path.resolve()}")

In [ ]:
from mrkdwn_analysis import MarkdownAnalyzer

analyzer = MarkdownAnalyzer(r"C:\Programming\multi-agent-system\src\multi_agent_system\extraction\gpt4_1_mini_rechnung_output.md")

In [ ]:
print(analyzer.analyse())

In [ ]:
print(analyzer.identify_tables())

In [ ]:
from pathlib import Path


# Markdown-Datei speichern
markdown_path = Path("local_rechnung_output.md")
markdown_path.write_text(markdown_output, encoding="utf-8")

print(f"Markdown gespeichert unter: {markdown_path.resolve()}")

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    VlmConvertOptions,
    VlmPipelineOptions,
)
from docling.datamodel.pipeline_options_vlm_model import ResponseFormat
from docling.datamodel.stage_model_specs import VlmModelSpec
from docling.datamodel.vlm_engine_options import (
    ApiVlmEngineOptions,
    VlmEngineType,
)
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)
from docling.pipeline.vlm_pipeline import VlmPipeline


# --------------------------------------------------
# 1. Umgebungsvariablen laden
# --------------------------------------------------

load_dotenv(find_dotenv())

openai_api_key = os.getenv("OPENAI_API_KEY")

if not openai_api_key:
    raise ValueError(
        "OPENAI_API_KEY wurde nicht gefunden. "
        "Bitte den API-Key in der .env-Datei setzen."
    )


# --------------------------------------------------
# 2. OpenAI-Modell
# --------------------------------------------------

#model_name = "gpt-5-nano"
#model_name = "gpt-5-mini"
model_name = "gpt-4.1-mini"


# --------------------------------------------------
# 3. VLM Model Spec
# --------------------------------------------------

model_spec = VlmModelSpec(
    name="OpenAI Document Parser",
    default_repo_id=model_name,

    prompt=(
        "Convert this document page to clean Markdown. "
        "Preserve all relevant document content and structure, including "
        "headings, paragraphs, lists, tables, and footnotes. "
    ),

    response_format=ResponseFormat.MARKDOWN,
)


# --------------------------------------------------
# 4. OpenAI Remote-VLM konfigurieren
# --------------------------------------------------

engine_options = ApiVlmEngineOptions(
    engine_type=VlmEngineType.API_OPENAI,

    headers={
        "Authorization": f"Bearer {openai_api_key}",
    },

    params={
        "model": model_name,

        # GPT-5-Modelle verwenden max_completion_tokens
        # statt max_tokens.
        "max_completion_tokens": 8192,

        # Überschreibt Doclings internes temperature=0.0.
        # Bei GPT-5 ist der Standardwert 1.0 kompatibel.
        "temperature": 1.0,
    },

    timeout=120,
    concurrency=2,
)


# --------------------------------------------------
# 5. VLM-Konfiguration
# --------------------------------------------------

vlm_options = VlmConvertOptions(
    model_spec=model_spec,
    engine_options=engine_options,
    batch_size=1,
)


pipeline_options = VlmPipelineOptions(
    vlm_options=vlm_options,

    # Explizites Opt-in, da Dokumentseiten
    # an die OpenAI API übertragen werden.
    enable_remote_services=True,
)


# --------------------------------------------------
# 6. DocumentConverter
# --------------------------------------------------

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_cls=VlmPipeline,
            pipeline_options=pipeline_options,
        )
    }
)


In [ ]:
# --------------------------------------------------
# 6. PDF konvertieren
# --------------------------------------------------

result = converter.convert("rechnung.pdf")


In [ ]:
# --------------------------------------------------
# 7. Markdown exportieren
# --------------------------------------------------

markdown = result.document.export_to_markdown()

print(markdown)

In [ ]:
import re
from pathlib import Path


# Markdown-Datei speichern
markdown_path = Path("gpt4_1_mini_rechnung_output.md")
markdown_path.write_text(markdown, encoding="utf-8")

print(f"Markdown gespeichert unter: {markdown_path.resolve()}")
